# SpendDNA
**Name:** Harshil Chauhan  
**Batch:** June 2024  


## Feature 1: Transaction Parser

In [21]:
import pandas as pd
import numpy as np

file_name = input('Enter the statement file name: ')

acct_name = input('Enter account holder name: ')
if acct_name.strip() == '':
    acct_name = 'Account Holder'

df = pd.read_csv(file_name)

# date column has 4 different formats mixed in, checking which one it is first
def fix_date(txt):
    txt = txt.strip()
    if '-' in txt and txt.split('-')[0].isdigit() and len(txt.split('-')[0]) == 4:
        return pd.to_datetime(txt, format='%Y-%m-%d', errors='coerce')
    if '/' in txt:
        return pd.to_datetime(txt, format='%d/%m/%y', errors='coerce')
    if '-' in txt:
        return pd.to_datetime(txt, format='%d-%b-%y', errors='coerce')
    return pd.to_datetime(txt, format='%d %b %Y', errors='coerce')

df['date'] = df['Date'].apply(fix_date)

# removing rupee sign, Rs. and commas from amount so it can become a number
amt = df['Amount'].astype(str).str.replace('₹', '', regex=False)
amt = amt.str.replace('Rs.', '', regex=False)
amt = amt.str.replace(',', '', regex=False)
df['amt'] = pd.to_numeric(amt.str.strip(), errors='coerce')

# DR/CR and Debit/Credit both show up, making it one format
df['kind'] = df['Type'].str.lower().replace({'dr': 'debit', 'cr': 'credit'})

dupes = df.duplicated().sum()
df = df.drop_duplicates()

bad_dt = df['date'].isna().sum()
bad_amt = df['amt'].isna().sum()
df = df.dropna(subset=['date', 'amt'])

# trynna find out the start and end date
period_start = df['date'].min()
period_end = df['date'].max()
# rough month count just by looking at year+month
num_months = (period_end.year - period_start.year) * 12 + (period_end.month - period_start.month) + 1
period_str = period_start.strftime('%d %b %Y') + ' to ' + period_end.strftime('%d %b %Y')

print('Parsed', len(df), 'transactions across', num_months, 'months (' + period_str + ').')
print('Dropped', dupes, 'duplicates.')
print(bad_amt, 'unparseable amounts,', bad_dt, 'unparseable dates.')


Enter the statement file name: DADSJUNE.csv
Enter account holder name: Pikachu
Parsed 1310 transactions across 6 months (01 Jan 2024 to 30 Jun 2024).
Dropped 18 duplicates.
0 unparseable amounts, 0 unparseable dates.


## Feature 2: Vendor Extractor

In [22]:
# mapping keywords to a clean vendor name
# order matters - specific ones like instamart need to come before plain swiggy
v_map = [
    ('Swiggy Instamart', ['SWIGGY-INSTAMART', 'SWIGGYINSTAMART', 'BUNDL TECH-INSTAMART', 'INSTAMART']),
    ('Blinkit', ['BLINKIT', 'GROFERS']),
    ('Zepto', ['ZEPTO', 'KIRANAKART']),
    ('BigBasket', ['BIGBASKET']),
    ('DMart', ['DMART', 'AVENUE SUPERMART']),
    ('Swiggy', ['SWIGGY', 'BUNDL']),
    ('Zomato', ['ZOMATO']),
    ('Amazon Prime', ['AMAZON PRIME', 'AMZN PRIME', 'AMAZON-PRIME']),
    ('Amazon', ['AMAZON', 'AMZN']),
    ('Flipkart', ['FLIPKART', 'FKART']),
    ('Myntra', ['MYNTRA']),
    ('Nykaa', ['NYKAA', 'FSN E-COMMERCE', 'INNOVATIVE RETAIL']),
    ('Ola', ['OLA', 'ANI TECHNOLOGIES']),
    ('Rapido', ['RAPIDO', 'ROPPEN']),
    ('Uber', ['UBER']),
    ('BMTC', ['BMTC', 'TUMMOC']),
    ('Cafe Coffee Day', ['COFFEE DAY', 'CCD']),
    ('Starbucks', ['STARBUCKS']),
    ('Third Wave Coffee', ['THIRDWAVE', 'THIRD WAVE', 'TWC']),
    ('Empire Restaurant', ['EMPIRE RESTAURANT']),
    ('Meghana Foods', ['MEGHANA']),
    ('Truffles', ['TRUFFLES']),
    ('Dineout', ['DINEOUT']),
    ('Restaurant', ['RESTAURANT']),
    ('Netflix', ['NETFLIX']),
    ('Hotstar', ['HOTSTAR', 'STAR INDIA']),
    ('Spotify', ['SPOTIFY']),
    ('Airtel', ['AIRTEL']),
    ('Vi', ['VI POSTPAID', 'VODAFONE', 'VI-RECHARGE']),
    ('Jio', ['JIO']),
    ('BESCOM', ['BESCOM', 'BANGALORE ELEC']),
    ('BWSSB', ['BWSSB']),
    ('Groww', ['GROWW', 'NEXTBILLION']),
    ('Zerodha', ['ZERODHA']),
    ('BPCL', ['BPCL']),
    ('HP Petrol', ['HP PETROL']),
    ('Indian Oil', ['INDIAN OIL', 'IOC']),
    ('BookMyShow', ['BOOKMYSHOW', 'BMS MOVIE', 'BIGTREE']),
    ('Rent', ['RENT-LANDLORD']),
    ('Salary', ['SALARY']),
    ('P2P Transfer', ['UPI-AMAN', 'UPI-ANKIT', 'UPI-KARAN', 'UPI-NEHA', 'UPI-PRIYA', 'UPI-SNEHA', 'UPI-VIKAS']),
    ('Cash Withdrawal', ['ATM-WDL', 'ATM WDL']),
]

# goes down the list top to bottom and stops at the first keyword that matches
def get_vendor(desc):
    d = desc.upper()
    for name, keys in v_map:
        for k in keys:
            if k in d:
                return name
    return 'Uncategorised'

df['vendor'] = df['Description'].apply(get_vendor)

print('Found', df['vendor'].nunique(), 'canonical vendors')
print()
print(df['vendor'].value_counts().head(10))


Found 42 canonical vendors

vendor
Swiggy              176
Zomato              121
Ola                  87
Amazon               76
Uber                 71
Zepto                71
Swiggy Instamart     67
Rapido               55
Blinkit              55
Flipkart             47
Name: count, dtype: int64


## Feature 3: Category Tagger

In [23]:
c_map = {
    'Swiggy': 'Food Delivery', 'Zomato': 'Food Delivery',
    'Blinkit': 'Quick Commerce', 'Zepto': 'Quick Commerce', 'Swiggy Instamart': 'Quick Commerce',
    'Amazon': 'Ecommerce', 'Flipkart': 'Ecommerce', 'Myntra': 'Ecommerce', 'Nykaa': 'Ecommerce',
    'Ola': 'Transport', 'Uber': 'Transport', 'Rapido': 'Transport', 'BMTC': 'Transport',
    'Cafe Coffee Day': 'Cafe', 'Starbucks': 'Cafe', 'Third Wave Coffee': 'Cafe',
    'Restaurant': 'Restaurants', 'Empire Restaurant': 'Restaurants', 'Meghana Foods': 'Restaurants',
    'Truffles': 'Restaurants', 'Dineout': 'Restaurants',
    'Amazon Prime': 'Subscriptions', 'Netflix': 'Subscriptions', 'Hotstar': 'Subscriptions', 'Spotify': 'Subscriptions',
    'Airtel': 'Utilities', 'Vi': 'Utilities', 'Jio': 'Utilities', 'BESCOM': 'Utilities', 'BWSSB': 'Utilities',
    'BigBasket': 'Groceries', 'DMart': 'Groceries',
    'Groww': 'Investments', 'Zerodha': 'Investments',
    'BPCL': 'Fuel', 'HP Petrol': 'Fuel', 'Indian Oil': 'Fuel',
    'BookMyShow': 'Entertainment',
    'P2P Transfer': 'Personal Transfer',
    'Cash Withdrawal': 'Cash Withdrawal',
    'Rent': 'Rent',
    'Salary': 'Income',
}

df['category'] = df['vendor'].map(c_map)

print(df['category'].value_counts())


category
Food Delivery        297
Transport            250
Quick Commerce       193
Ecommerce            170
Cafe                  99
Restaurants           73
Utilities             43
Subscriptions         41
Groceries             33
Fuel                  28
Investments           23
Personal Transfer     18
Cash Withdrawal       17
Entertainment         13
Income                 6
Rent                   6
Name: count, dtype: int64


## Feature 4: Spending Overview

In [24]:
deb = df[df['kind'] == 'debit']
cred = df[df['kind'] == 'credit']

tot_cr = cred['amt'].sum()
tot_dr = deb['amt'].sum()
net = tot_cr - tot_dr
sav_rate = (net / tot_cr) * 100

# transfers, cash withdrawals and rent aren't really "spending" so keeping them out of the %
spend = deb[~deb['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent'])]
spend_total = spend['amt'].sum()

# adding up amt for every category, then sorting biggest first
cat_amt = spend.groupby('category')['amt'].sum().sort_values(ascending=False)
cat_pct = (cat_amt / spend_total) * 100

vnd_amt = spend.groupby('vendor')['amt'].sum().sort_values(ascending=False)
vnd_cnt = spend.groupby('vendor')['amt'].count()  # how many orders per vendor

print('=' * 60)
print('EXECUTIVE SUMMARY')
print('=' * 60)
print('Total credits  : Rs. ' + format(int(tot_cr), ','))
print('Total debits   : Rs. ' + format(int(tot_dr), ','))
print('Net change     : Rs. ' + format(int(net), ','))
print('Savings rate   : ' + str(round(sav_rate, 1)) + '%')
print('Transactions   : ' + str(len(df)))
print('Unique vendors : ' + str(df['vendor'].nunique()))
print()

print('TOP 5 CATEGORIES (% of spending)')
print('-' * 60)
top5_cat = cat_pct.head(5)  # head() since it is already sorted big to small
for name in top5_cat.index:
    pct = cat_pct[name]
    bar = chr(9608) * int(pct / 2)
    print(name.ljust(16) + bar.ljust(22) + str(round(pct, 1)).rjust(5) + '%  Rs. ' + format(int(cat_amt[name]), ','))
print()

print('TOP 5 VENDORS (by spend)')
print('-' * 60)
top5_vnd = vnd_amt.head(5)
for name in top5_vnd.index:
    print(name.ljust(16) + 'Rs. ' + format(int(vnd_amt[name]), ',').rjust(10) + '  (' + str(vnd_cnt[name]) + ' orders)')


EXECUTIVE SUMMARY
Total credits  : Rs. 509,774
Total debits   : Rs. 1,678,901
Net change     : Rs. -1,169,127
Savings rate   : -229.3%
Transactions   : 1310
Unique vendors : 42

TOP 5 CATEGORIES (% of spending)
------------------------------------------------------------
Ecommerce       ████████████████████   40.2%  Rs. 602,968
Investments     ████████               16.5%  Rs. 248,160
Food Delivery   ████                    8.6%  Rs. 129,054
Restaurants     ███                     7.8%  Rs. 117,737
Quick Commerce  ███                     6.4%  Rs. 95,667

TOP 5 VENDORS (by spend)
------------------------------------------------------------
Amazon          Rs.    318,422  (76 orders)
Zerodha         Rs.    210,000  (14 orders)
Flipkart        Rs.    177,510  (47 orders)
Swiggy          Rs.     73,738  (176 orders)
Myntra          Rs.     69,529  (20 orders)


## Feature 5: Monthly Trend Analysis

In [25]:
df['month'] = df['date'].dt.month
spend2 = df[(df['kind'] == 'debit') & (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

# rows = category, columns = month, values = total spent that month
# pivot_table makes a grid: one row per category, one column per month number
m_tab = spend2.pivot_table(values='amt', index='category', columns='month', aggfunc='sum', fill_value=0)
mon_name = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun'}

# comparing first month to last month for each category to see growth
grow = {}
for c in m_tab.index:
    row = m_tab.loc[c]
    # iloc[0] is the first month in the row, iloc[-1] is the last month
    if row.iloc[0] == 0:
        continue
    grow[c] = ((row.iloc[-1] - row.iloc[0]) / row.iloc[0]) * 100

up_cat = max(grow, key=lambda x: grow[x])
down_cat = min(grow, key=lambda x: grow[x])

print('=' * 60)
print('MONTHLY TREND (Food Delivery)')
print('=' * 60)
food_row = m_tab.loc['Food Delivery']
max_food = food_row.max()
for m in food_row.index:
    val = food_row[m]
    bar = chr(9608) * int((val / max_food) * 20)
    print(mon_name[m] + '  Rs. ' + format(int(val), ',').rjust(8) + '  ' + bar)
print()
print('TRENDING UP THE MOST   : ' + up_cat + ' (' + str(round(grow[up_cat], 1)) + '% change)')
print('TRENDING DOWN THE MOST : ' + down_cat + ' (' + str(round(grow[down_cat], 1)) + '% change)')


MONTHLY TREND (Food Delivery)
Jan  Rs.   20,890  ██████████████████
Feb  Rs.   21,452  ██████████████████
Mar  Rs.   20,850  ██████████████████
Apr  Rs.   23,054  ████████████████████
May  Rs.   22,167  ███████████████████
Jun  Rs.   20,641  █████████████████

TRENDING UP THE MOST   : Cafe (57.2% change)
TRENDING DOWN THE MOST : Fuel (-90.5% change)


## Feature 6: Time-of-Day Patterns

In [26]:
df['hour'] = df['Time'].str[:2].astype(int)
spend3 = df[(df['kind'] == 'debit') & (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))]

# trynna check food delivery late night (9 PM to 2 AM)
food = spend3[spend3['category'] == 'Food Delivery']
late = food[(food['hour'] >= 21) | (food['hour'] < 2)]
late_pct = (len(late) / len(food)) * 100

# trynna check cafe morning (8 to 11 AM)
cafe = spend3[spend3['category'] == 'Cafe']
morning = cafe[(cafe['hour'] >= 8) & (cafe['hour'] < 11)]
morn_pct = (len(morning) / len(cafe)) * 100

print('=' * 60)
print('TIME-OF-DAY PATTERNS')
print('=' * 60)
print('Food Delivery late-night (9 PM - 2 AM): ' + str(round(late_pct, 1)) + '%')
print('Cafe spend in the morning (8-11 AM)    : ' + str(round(morn_pct, 1)) + '%')
print()

# building a category x hour grid with numpy, 3 hour blocks
# giving every category a row number so it can go into the numpy grid
cats = sorted(spend3['category'].unique())
idx = {}
for i in range(len(cats)):
    idx[cats[i]] = i

mat = np.zeros((len(cats), 24), dtype=int)  # rows = category, columns = hour 0-23
for i in range(len(spend3)):
    row = spend3.iloc[i]
    mat[idx[row['category']]][row['hour']] += 1  # +1 to that category's hour slot

print('HOURLY ACTIVITY GRID (3-hour blocks)')
print('-' * 60)
hdr = '                  '
for h in range(0, 24, 3):
    hdr = hdr + str(h).zfill(2) + '  '
print(hdr)
for i in range(len(cats)):
    row = mat[i]
    peak = row.max()
    line = cats[i].ljust(18)
    for h in range(0, 24, 3):
        chunk = sum(row[h:h + 3])  # adding 3 hours together so the grid isn't 24 columns wide
        if peak == 0 or chunk == 0:
            block = '.'
        else:
            r = chunk / (peak * 3)
            if r < 0.25:
                block = chr(9617)
            elif r < 0.5:
                block = chr(9618)
            else:
                block = chr(9619)
        line = line + block + '   '
    print(line)


TIME-OF-DAY PATTERNS
Food Delivery late-night (9 PM - 2 AM): 20.5%
Cafe spend in the morning (8-11 AM)    : 29.3%

HOURLY ACTIVITY GRID (3-hour blocks)
------------------------------------------------------------
                  00  03  06  09  12  15  18  21  
Cafe              ░   ░   ▒   ▓   ▒   ▓   ▒   ░   
Ecommerce         ▓   ▓   ▒   ▓   ▓   ▓   ▓   ▓   
Entertainment     ▒   .   ▓   .   .   ▒   ▓   ░   
Food Delivery     ░   ░   ░   ▒   ▒   ▒   ▓   ▒   
Fuel              ▒   ▒   ░   ▒   ▓   ▓   ▒   ▒   
Groceries         ▒   ▒   ▒   ▓   ▒   .   ▒   ▒   
Investments       ░   ░   ░   ▒   ░   .   ░   ░   
Quick Commerce    ░   ░   ░   ▒   ▓   ▒   ▓   ▓   
Restaurants       ░   ░   ░   ░   ▒   ▒   ▓   ░   
Subscriptions     ░   ▓   ░   ░   ░   .   ░   ░   
Transport         ░   ░   ▓   ▓   ▓   ▓   ▓   ▒   
Utilities         ▓   ░   ▓   ▒   ▓   ▒   ▓   ░   


## Feature 7: Anomaly Detection

In [27]:
# z-score tells us how far a transaction is from the average for its own category
spend4 = df[(df['kind'] == 'debit') & (~df['category'].isin(['Personal Transfer', 'Cash Withdrawal', 'Rent']))].copy()

# transform keeps one value per row (not one per category) so it lines up with amt directly
cat_mean = spend4.groupby('category')['amt'].transform('mean')
cat_std = spend4.groupby('category')['amt'].transform('std')
spend4['z'] = (spend4['amt'] - cat_mean) / cat_std

odd = spend4[spend4['z'] > 2].sort_values('z', ascending=False)

print('=' * 60)
print('TOP ANOMALIES (z-score above 2)')
print('=' * 60)
print('Found', len(odd), 'anomalous transactions')
print()
top_odd = odd.head(5)
for i in range(len(top_odd)):
    row = top_odd.iloc[i]
    day = row['date'].strftime('%d %b')
    print(day + ' - ' + row['vendor'].ljust(16) + 'Rs. ' + format(int(row['amt']), ',').rjust(8) + '  (z=' + str(round(row['z'], 1)) + ')')


TOP ANOMALIES (z-score above 2)
Found 22 anomalous transactions

26 Jun - Amazon          Rs.   22,008  (z=4.1)
07 Feb - Amazon          Rs.   21,986  (z=4.1)
26 Feb - Restaurant      Rs.    8,383  (z=3.9)
22 Jun - Dineout         Rs.    7,935  (z=3.6)
31 Mar - Meghana Foods   Rs.    7,931  (z=3.6)


## Feature 8: Spending Archetype Detection

In [28]:
# one function per rule, each one just checks a threshold from the brief

def is_foodie(p):
    v = p.get('Food Delivery', 0) + p.get('Restaurants', 0) + p.get('Cafe', 0)
    return v > 25, v

def is_qcom(p):
    v = p.get('Quick Commerce', 0)
    return v > 15, v

def is_shopaholic(p):
    v = p.get('Ecommerce', 0)
    return v > 15, v

def is_investor(p):
    v = p.get('Investments', 0)
    return v > 15, v

# reusing the food delivery rows already worked out in feature 6
def is_late_snacker(food_df):
    late = food_df[(food_df['hour'] >= 21) | (food_df['hour'] < 2)]
    v = (len(late) / len(food_df)) * 100
    return v > 50, v

def is_cab_commuter(p):
    v = p.get('Transport', 0)
    return v > 10, v

def is_sub_lover(all_df):
    v = all_df[all_df['category'] == 'Subscriptions']['vendor'].nunique()
    return v >= 5, v

def is_yolo(rate):
    return rate < 10, rate

def is_disciplined(rate):
    return rate > 40, rate

# each result is a (True/False, number) pair - the number is what gets printed later
r_foodie = is_foodie(cat_pct)
r_qcom = is_qcom(cat_pct)
r_shop = is_shopaholic(cat_pct)
r_inv = is_investor(cat_pct)
r_late = is_late_snacker(food)
r_cab = is_cab_commuter(cat_pct)
r_sub = is_sub_lover(deb)
r_yolo = is_yolo(sav_rate)
r_disc = is_disciplined(sav_rate)

print('=' * 60)
print(acct_name.upper() + "'S SPENDING ARCHETYPES")
print('=' * 60)
if r_foodie[0]:
    print('-> THE FOODIE               (' + str(round(r_foodie[1], 1)) + '% on food)')
if r_qcom[0]:
    print('-> THE QUICK COMMERCE JUNKIE (' + str(round(r_qcom[1], 1)) + '% on Q-com)')
if r_shop[0]:
    print('-> THE SHOPAHOLIC            (' + str(round(r_shop[1], 1)) + '% on e-commerce)')
if r_inv[0]:
    print('-> THE INVESTOR              (' + str(round(r_inv[1], 1)) + '% on SIPs)')
if r_late[0]:
    print('-> THE LATE-NIGHT SNACKER    (' + str(round(r_late[1], 1)) + '% food after 9 PM)')
if r_cab[0]:
    print('-> THE CAB COMMUTER          (' + str(round(r_cab[1], 1)) + '% on transport)')
if r_sub[0]:
    print('-> THE SUBSCRIPTION LOVER    (' + str(r_sub[1]) + ' active subs)')
if r_yolo[0]:
    print('-> THE YOLO SPENDER          (savings rate ' + str(round(r_yolo[1], 1)) + '%)')
if r_disc[0]:
    print('-> THE DISCIPLINED SAVER     (savings rate ' + str(round(r_disc[1], 1)) + '%)')


PIKACHU'S SPENDING ARCHETYPES
-> THE SHOPAHOLIC            (40.2% on e-commerce)
-> THE INVESTOR              (16.5% on SIPs)
-> THE YOLO SPENDER          (savings rate -229.3%)


## Bonus: Vendor Cleanup Audit

In [29]:
# checking if any description slipped through without a vendor match
missed = df[df['vendor'] == 'Uncategorised']
print('Uncategorised transactions:', len(missed))
if len(missed) > 0:
    print(missed['Description'].unique())
else:
    print('None - every description got mapped to a vendor.')


Uncategorised transactions: 0
None - every description got mapped to a vendor.


## Final Report

In [30]:
print()
print('=' * 64)
print(' SpendDNA REPORT - ' + acct_name.upper())
print(' ' + str(num_months) + ' months - ' + str(len(df)) + ' transactions - ' + period_str)
print('=' * 64)
print()

print('EXECUTIVE SUMMARY')
print('-' * 64)
print('Total credits  : Rs. ' + format(int(tot_cr), ','))
print('Total debits   : Rs. ' + format(int(tot_dr), ','))
print('Net change     : Rs. ' + format(int(net), ','))
print('Savings rate   : ' + str(round(sav_rate, 1)) + '%')
print('Transactions   : ' + str(len(df)))
print('Unique vendors : ' + str(df['vendor'].nunique()))
print()

print('TOP CATEGORIES (% of spending)')
print('-' * 64)
for name in top5_cat.index:
    pct = cat_pct[name]
    bar = chr(9608) * int(pct / 2)
    print(name.ljust(16) + bar.ljust(22) + str(round(pct, 1)).rjust(5) + '%  Rs. ' + format(int(cat_amt[name]), ','))
print()

print('TOP VENDORS')
print('-' * 64)
for name in top5_vnd.index:
    print(name.ljust(16) + 'Rs. ' + format(int(vnd_amt[name]), ',').rjust(10) + '  (' + str(vnd_cnt[name]) + ' orders)')
print()

print('TIME-OF-DAY PATTERNS')
print('-' * 64)
print('Food Delivery late-night (9 PM - 2 AM): ' + str(round(late_pct, 1)) + '%')
print('Cafe morning visits (8-11 AM)          : ' + str(round(morn_pct, 1)) + '%')
print()

print('MONTHLY TREND (Food Delivery)')
print('-' * 64)
for m in food_row.index:
    val = food_row[m]
    bar = chr(9608) * int((val / max_food) * 20)
    print(mon_name[m] + '  Rs. ' + format(int(val), ',').rjust(8) + '  ' + bar)
print()

print('TOP ANOMALIES')
print('-' * 64)
for i in range(len(top_odd)):
    row = top_odd.iloc[i]
    day = row['date'].strftime('%d %b')
    print(day + ' - ' + row['vendor'].ljust(16) + 'Rs. ' + format(int(row['amt']), ',').rjust(8) + '  (z=' + str(round(row['z'], 1)) + ')')
print()

print(acct_name.upper() + "'S SPENDING ARCHETYPES")
print('-' * 64)
if r_foodie[0]:
    print('-> THE FOODIE               (' + str(round(r_foodie[1], 1)) + '% on food)')
if r_qcom[0]:
    print('-> THE QUICK COMMERCE JUNKIE (' + str(round(r_qcom[1], 1)) + '% on Q-com)')
if r_shop[0]:
    print('-> THE SHOPAHOLIC            (' + str(round(r_shop[1], 1)) + '% on e-commerce)')
if r_inv[0]:
    print('-> THE INVESTOR              (' + str(round(r_inv[1], 1)) + '% on SIPs)')
if r_late[0]:
    print('-> THE LATE-NIGHT SNACKER    (' + str(round(r_late[1], 1)) + '% food after 9 PM)')
if r_cab[0]:
    print('-> THE CAB COMMUTER          (' + str(round(r_cab[1], 1)) + '% on transport)')
if r_sub[0]:
    print('-> THE SUBSCRIPTION LOVER    (' + str(r_sub[1]) + ' active subs)')
if r_yolo[0]:
    print('-> THE YOLO SPENDER          (savings rate ' + str(round(r_yolo[1], 1)) + '%)')
if r_disc[0]:
    print('-> THE DISCIPLINED SAVER     (savings rate ' + str(round(r_disc[1], 1)) + '%)')
print()

print('=' * 64)
print(' KEY INSIGHTS')
print('=' * 64)
monthly_burn = abs(net) / num_months
print(' 1. ' + acct_name + ' is burning through Rs. ' + format(int(monthly_burn), ',') + ' per month more')
print('    than earned, a savings rate of ' + str(round(sav_rate, 1)) + '%.')
print()
print(' 2. Ecommerce alone eats up ' + str(round(cat_pct.get('Ecommerce', 0), 1)) + '% of the spending,')
print('    the single biggest slice of the wallet.')
print()
top1 = top_odd.iloc[0]
print(' 3. The biggest single anomaly was a Rs. ' + format(int(top1['amt']), ',') + ' charge at')
print('    ' + top1['vendor'] + ', way outside the usual pattern for that category.')
print('=' * 64)



 SpendDNA REPORT - PIKACHU
 6 months - 1310 transactions - 01 Jan 2024 to 30 Jun 2024

EXECUTIVE SUMMARY
----------------------------------------------------------------
Total credits  : Rs. 509,774
Total debits   : Rs. 1,678,901
Net change     : Rs. -1,169,127
Savings rate   : -229.3%
Transactions   : 1310
Unique vendors : 42

TOP CATEGORIES (% of spending)
----------------------------------------------------------------
Ecommerce       ████████████████████   40.2%  Rs. 602,968
Investments     ████████               16.5%  Rs. 248,160
Food Delivery   ████                    8.6%  Rs. 129,054
Restaurants     ███                     7.8%  Rs. 117,737
Quick Commerce  ███                     6.4%  Rs. 95,667

TOP VENDORS
----------------------------------------------------------------
Amazon          Rs.    318,422  (76 orders)
Zerodha         Rs.    210,000  (14 orders)
Flipkart        Rs.    177,510  (47 orders)
Swiggy          Rs.     73,738  (176 orders)
Myntra          Rs.     69,52

## Reflection

**Hardest part:** The vendor dictionary. Some names like `BUNDL Tech P L` and `KIRANAKART TECH` are actually the parent companies of Swiggy and Zepto, so I had to look those up before mapping them right. Date parsing was also annoying since a plain `pd.to_datetime(dayfirst=True)` on the whole column quietly mixed up some of the ISO-format dates.

**What I would do differently:** Check `df['Description'].unique()` more carefully before writing the vendor dictionary, a few tricky vendor names almost slipped through as Uncategorised.

**Constraints followed:**
- No pandas-profiling, matplotlib, seaborn, or plotly
- No scikit-learn, scipy.stats, or statsmodels — z-score written by hand
- No collections.Counter — used plain dictionaries and pandas groupby instead
- No regex — all matching done with `in` and `.str.contains`
- Only pandas, NumPy, and datetime
